# KASA-42 — manifest only

**One job: produce the four files the H200 needs, and nothing else.**

```
results/manifest.parquet   metadata for 1.4M segments across 42 languages
results/splits.json        book-disjoint train/dev/test
results/vocab.json         shared CTC vocab, DONDO's 49 tokens extended
results/mixture.json       temperature-sampled training mixture
```

### Settings → Accelerator: **None (CPU)**

Deliberately no GPU. This step reads only parquet *metadata* columns — it never
touches audio and has no use for a GPU. A CPU session gets roughly **30 GB RAM
instead of ~13 GB**, which is the resource that actually ran out, and it leaves
your GPU quota intact for the rehearsal.

### Settings → Internet: **On**

Resumable: each config is written to `results/manifest_parts/` as it completes,
and a re-run skips whatever is already there. If the session dies, re-run — you
never start over.

**Save Version (Quick Save) when finished**, or Kaggle discards the output.

In [ ]:
# Must precede every other import: huggingface_hub reads these at import time.
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Optional: Add-ons -> Secrets -> HF_TOKEN lifts rate limits.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    print('No HF_TOKEN — rate-limited, but this step is small so it is fine.')

In [ ]:
import sys, subprocess, pathlib

WORK = pathlib.Path('/kaggle/working/kasa42')
if WORK.exists():
    subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/NasamuAlhassan/kasa42.git', str(WORK)], check=True)

os.chdir(WORK)
if str(WORK / 'src') not in sys.path:
    sys.path.insert(0, str(WORK / 'src'))
for name in [m for m in sys.modules if m.startswith('kasa42')]:
    del sys.modules[name]

print('rev', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                            capture_output=True, text=True).stdout.strip())

import psutil
print(f'RAM: {psutil.virtual_memory().total/1e9:.0f} GB total, '
      f'{psutil.virtual_memory().available/1e9:.0f} GB available')
done = len(list((WORK / 'results/manifest_parts').glob('*.parquet'))) \
    if (WORK / 'results/manifest_parts').exists() else 0
print(f'{done}/42 configs already done')

## The manifest

Re-run this cell as many times as needed. It skips completed configs.

`--workers 4` is conservative on purpose. Each worker buffers ~4 MB blocks, and
shards are streamed in 2048-row batches rather than read whole, so peak memory
stays bounded. Raise it only if RAM is clearly untroubled.

In [ ]:
!HF_HUB_DISABLE_XET=1 python -m kasa42.data.build_manifest --workers 4

In [ ]:
import pathlib
parts = sorted(pathlib.Path('results/manifest_parts').glob('*.parquet'))
print(f'{len(parts)}/42 configs\n')
for p in parts:
    print(f'  {p.stem:28s} {p.stat().st_size/1e6:7.2f} MB')
if len(parts) < 42:
    have = {p.stem for p in parts}
    print('\nRe-run the cell above to fetch the rest.')

## Splits, vocab, mixture — the first real numbers

In [ ]:
!python -m kasa42.data.splits

In [ ]:
!python -m kasa42.data.vocab

In [ ]:
# 400 h x 2 epochs, not 700 x 3: the T4 benchmark projected ~26 h on the H200,
# and trimming the budget mostly takes hours off the largest languages that
# temperature sampling was already capping. The tail keeps its data.
!python -m kasa42.data.mixture --alpha 0.5 --cap-hours 40 --budget-hours 400

In [ ]:
import pathlib
need = ['results/manifest.parquet', 'results/splits.json',
        'results/vocab.json', 'results/mixture.json']
ok = True
for f in need:
    p = pathlib.Path(f)
    if p.exists():
        print(f'ok       {f:32s} {p.stat().st_size/1e6:8.2f} MB')
    else:
        print(f'MISSING  {f}')
        ok = False

if ok:
    print('\nAll four present.')
    print('1. Save Version (Quick Save) so Kaggle keeps the output')
    print('2. Download them from the Output pane')
    print('3. Commit to the repo, so Thursday starts with data prep done')